<a href="https://colab.research.google.com/github/rj-Fariha/document-extraction-audit/blob/main/Document_extraction_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Document Extraction & Quality Audit System
# Rule-based extraction + Gradient Boosting quality classifier
# (Same boosting technique as published thesis - applied here
#  to a new problem: automated document QA)
#
# Runs 100% locally. No API keys, no internet calls, no cost,
# no downloads. Works on any machine, any time, forever.
# ============================================================

import re
import random
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

random.seed(42)

# ------------------------------------------------------------
# STEP 1: Generate realistic OCR-style receipt text
# (Simulates the messy, noisy output real OCR produces —
#  including missing chars, misread digits, extra whitespace)
# ------------------------------------------------------------
VENDORS = ["NOVAMART RETAIL", "CITY SUPERMARKET", "AL FUTTAIM ELECTRONICS",
           "QUICK MART", "GREEN VALLEY GROCERY", "TECH ZONE LLC"]

def make_synthetic_receipt(add_noise: bool):
    vendor = random.choice(VENDORS)
    day = random.randint(1, 28)
    month = random.randint(1, 12)
    year = random.choice([2025, 2026])
    total = round(random.uniform(20, 900), 2)
    tax = round(total * 0.05, 2)
    currency = "AED"

    text = f"{vendor}\nDate: {day:02d}/{month:02d}/{year}\n"
    text += f"Subtotal: {total - tax:.2f}\nVAT: {tax:.2f}\n"
    text += f"TOTAL: {total:.2f} {currency}\nThank you for shopping"

    ground_truth = {
        "vendor_name": vendor, "date": f"{year}-{month:02d}-{day:02d}",
        "total_amount": total, "currency": currency,
    }

    if add_noise:
        # simulate common OCR mistakes: digit confusion, missing chars, extra symbols
        text = text.replace("O", "0").replace("l", "1")
        text = re.sub(r"(\d)", lambda m: m.group(1) if random.random() > 0.08
                       else random.choice("0123456789"), text)
        if random.random() > 0.7:
            text = text.replace("TOTAL:", "T0TAL ")  # sometimes the keyword itself gets mangled

    return text, ground_truth

# Build a labeled dataset: 150 clean, 150 noisy receipts
dataset = []
for _ in range(150):
    text, gt = make_synthetic_receipt(add_noise=False)
    dataset.append({"text": text, "ground_truth": gt, "is_noisy": False})
for _ in range(150):
    text, gt = make_synthetic_receipt(add_noise=True)
    dataset.append({"text": text, "ground_truth": gt, "is_noisy": True})

random.shuffle(dataset)
print(f"Generated {len(dataset)} synthetic receipts ({sum(d['is_noisy'] for d in dataset)} noisy)")
print("\n--- Sample receipt ---")
print(dataset[0]["text"])

Generated 300 synthetic receipts (150 noisy)

--- Sample receipt ---
AL FUTTAIM ELECTRONICS
Date: 26/01/2025
Subtotal: 602.64
VAT: 31.72
TOTAL: 634.36 AED
Thank you for shopping


In [ ]:
# ------------------------------------------------------------
# STEP 2: Rule-based field extraction (regex)
# This is the "extraction engine" - no LLM, no API, pure logic.
# ------------------------------------------------------------

def extract_fields(text: str) -> dict:
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    vendor_name = lines[0] if lines else "UNKNOWN"

    date_match = re.search(r"(\d{1,2})/(\d{1,2})/(\d{4})", text)
    date = f"{date_match.group(3)}-{date_match.group(2)}-{date_match.group(1)}" if date_match else None

    total_match = re.search(r"(?<!sub)T[O0]TAL[:\s]*([\d.]+)", text, re.IGNORECASE)
    total_amount = float(total_match.group(1)) if total_match else None

    currency_match = re.search(r"\b(AED|USD|EUR|GBP)\b", text)
    currency = currency_match.group(1) if currency_match else "UNKNOWN"

    return {
        "vendor_name": vendor_name,
        "date": date,
        "total_amount": total_amount,
        "currency": currency,
        # feature signals used later by the ML quality classifier
        "found_date": date is not None,
        "found_total": total_amount is not None,
        "found_currency": currency != "UNKNOWN",
        "text_length": len(text),
    }

# Run extraction on every sample
for row in dataset:
    row["extracted"] = extract_fields(row["text"])

print("--- Example extraction ---")
print("Ground truth:", dataset[0]["ground_truth"])
print("Extracted:   ", dataset[0]["extracted"])

--- Example extraction ---
Ground truth: {'vendor_name': 'AL FUTTAIM ELECTRONICS', 'date': '2025-01-26', 'total_amount': 634.36, 'currency': 'AED'}
Extracted:    {'vendor_name': 'AL FUTTAIM ELECTRONICS', 'date': '2025-01-26', 'total_amount': 634.36, 'currency': 'AED', 'found_date': True, 'found_total': True, 'found_currency': True, 'text_length': 108}


In [ ]:
# ------------------------------------------------------------
# STEP 3: Determine if each extraction was actually CORRECT
# (compare against ground truth - this becomes our training label)
# ------------------------------------------------------------

def is_extraction_correct(row) -> bool:
    gt = row["ground_truth"]
    ex = row["extracted"]
    total_ok = ex["total_amount"] is not None and abs(ex["total_amount"] - gt["total_amount"]) < 0.5
    date_ok = ex["date"] == gt["date"]
    return total_ok and date_ok

for row in dataset:
    row["is_correct"] = is_extraction_correct(row)

correct_count = sum(r["is_correct"] for r in dataset)
print(f"Rule-based extraction alone got {correct_count}/{len(dataset)} "
      f"({correct_count/len(dataset):.1%}) fully correct")

# ------------------------------------------------------------
# STEP 4: Train a Gradient Boosting classifier to PREDICT
# whether an extraction is trustworthy - using only signals
# available at extraction time (not the ground truth itself).
# This is the same boosting technique from the thesis, applied
# here as a quality-control layer on top of the rule-based engine.
# ------------------------------------------------------------

df = pd.DataFrame([{
    "found_date": r["extracted"]["found_date"],
    "found_total": r["extracted"]["found_total"],
    "found_currency": r["extracted"]["found_currency"],
    "text_length": r["extracted"]["text_length"],
    "is_correct": r["is_correct"],
} for r in dataset])

X = df[["found_date", "found_total", "found_currency", "text_length"]]
y = df["is_correct"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

clf = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("\n--- Quality Classifier Performance (held-out test set) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.1%}")
print(f"Precision: {precision_score(y_test, y_pred):.1%}")
print(f"Recall:    {recall_score(y_test, y_pred):.1%}")
print(f"F1 score:  {f1_score(y_test, y_pred):.1%}")

Rule-based extraction alone got 212/300 (70.7%) fully correct

--- Quality Classifier Performance (held-out test set) ---
Accuracy:  73.3%
Precision: 73.5%
Recall:    96.2%
F1 score:  83.3%


In [ ]:
# ------------------------------------------------------------
# STEP 5: Audit layer - cross-check extracted data against a
# reference "vendor master" and flag discrepancies with reasons.
# (This is the automated version of the manual QA process
#  performed daily at InsideMaps.)
# ------------------------------------------------------------

VENDOR_MASTER = {v: {"expected_currency": "AED", "max_reasonable_total": 1000.0} for v in VENDORS}

def audit(extracted: dict, predicted_reliable: bool) -> list[str]:
    issues = []
    vendor = extracted["vendor_name"]

    if vendor not in VENDOR_MASTER:
        issues.append(f"Vendor '{vendor}' not found in vendor master - possible OCR misread.")
        return issues

    expected = VENDOR_MASTER[vendor]
    if extracted["currency"] != expected["expected_currency"]:
        issues.append(f"Currency mismatch: got {extracted['currency']}, expected {expected['expected_currency']}.")

    if extracted["total_amount"] and extracted["total_amount"] > expected["max_reasonable_total"]:
        issues.append(f"Total {extracted['total_amount']} exceeds historical maximum - possible digit error.")

    if not predicted_reliable:
        issues.append("ML quality classifier flagged this extraction as low-confidence - recommend human review.")

    return issues

# Run the full pipeline end-to-end on the whole dataset
results = []
for row in dataset:
    features = pd.DataFrame([{
        "found_date": row["extracted"]["found_date"],
        "found_total": row["extracted"]["found_total"],
        "found_currency": row["extracted"]["found_currency"],
        "text_length": row["extracted"]["text_length"],
    }])
    predicted_reliable = bool(clf.predict(features)[0])
    issues = audit(row["extracted"], predicted_reliable)
    results.append({
        "vendor": row["extracted"]["vendor_name"],
        "total": row["extracted"]["total_amount"],
        "actually_correct": row["is_correct"],
        "ml_predicted_reliable": predicted_reliable,
        "num_issues_flagged": len(issues),
        "status": "flagged" if issues else "clean",
    })

results_df = pd.DataFrame(results)
print(results_df.head(15))
print(f"\nTotal flagged for review: {(results_df['status'] == 'flagged').sum()} / {len(results_df)}")

                    vendor   total  actually_correct  ml_predicted_reliable  \
0   AL FUTTAIM ELECTRONICS  634.36              True                   True   
1               QUICK MART  699.49             False                   True   
2               QUICK MART  392.25              True                   True   
3         CITY SUPERMARKET  885.86              True                   True   
4         CITY SUPERMARKET  788.24             False                   True   
5            TECH ZONE LLC  269.73              True                   True   
6   AL FUTTAIM ELECTRONICS  152.73              True                   True   
7          N0VAMART RETAIL  715.03             False                   True   
8               QUICK MART   76.52              True                   True   
9         CITY SUPERMARKET  349.53              True                   True   
10    GREEN VALLEY GROCERY  429.19              True                   True   
11              QUICK MART   21.90              True